In [9]:
!pip install openai




   ---------------------------------------- 0.0/567.4 kB ? eta -:--:--
   --------------------------------------- 567.4/567.4 kB 19.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 27.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

# Define the ConversationalLLM class for Flan-T5
class ConversationalLLM:
    """
    Wraps an instruction-tuned encoder-decoder model (Flan-T5) for conversational use.
    """
    def __init__(self, model_name: str = "google/flan-t5-base", max_length: int = 128, temperature: float = 0.3):
        self.model_name = model_name
        self.max_length = max_length
        self.temperature = temperature
        
        set_seed(42)
        print(f"Loading tokenizer for {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        
        print(f"Loading model {model_name}...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.model.to(self.device)

    def generate_response(self, instruction: str, conversation_history: str = "") -> str:
        """
        Generate a response based on the instruction.
        In this version, we simplify the prompt by not including additional history.
        """
        # Use a simplified prompt that directly instructs the model to rephrase the question.
        prompt = f"Rephrase the following onboarding question in a friendly and clear manner without adding extra content:\n\"{instruction}\""
        
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_length=self.max_length,
                temperature=self.temperature,
                top_p=0.9,
                do_sample=True,
                num_return_sequences=1
            )
        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        return generated_text.strip()

# Define the ConversationManager class
class ConversationManager:
    """
    Manages the conversation flow and user onboarding questions.
    """
    def __init__(self):
        self.required_questions = [
            "Please provide your full legal name.",
            "What is your primary phone number?",
            "What is your email address?",
            "Do you anticipate frequent international transactions?",
        ]
        self.current_question_index = 0
        self.answers = {}
        self.conversation_history = ""

    def get_next_question(self):
        if self.current_question_index < len(self.required_questions):
            return self.required_questions[self.current_question_index]
        return None

    def record_answer(self, question: str, answer: str):
        self.answers[question] = answer
        self.conversation_history += f"\nUser: {answer}"

    def advance_flow(self):
        self.current_question_index += 1

    def is_onboarding_complete(self) -> bool:
        return self.current_question_index >= len(self.required_questions)

# --- Conversation Loop ---
# Instantiate the LLM (Flan-T5) and the ConversationManager
llm = ConversationalLLM(model_name="google/flan-t5-base", max_length=128, temperature=0.3)
conv_manager = ConversationManager()

print("System: Onboarding conversation started.")
print("---------------------------------------")

while not conv_manager.is_onboarding_complete():
    # Retrieve next question
    question = conv_manager.get_next_question()
    if question is None:
        break
    
    # Use Flan-T5 to rephrase the question in a friendly manner.
    rephrased_question = llm.generate_response(instruction=question)
    
    # Print the AI-generated question
    print("Assistant (Question):", rephrased_question)
    
    # Capture user input in the notebook
    user_input = input("User (Answer): ")
    
    # Record the answer and advance the flow
    conv_manager.record_answer(question, user_input)
    conv_manager.advance_flow()

print("\nAssistant: Thank you! We have all the information we need for onboarding.")
print("Your collected answers:", conv_manager.answers)


Loading tokenizer for google/flan-t5-base...
Loading model google/flan-t5-base...
Using device: cpu
System: Onboarding conversation started.
---------------------------------------
Assistant (Question): "Please provide your full legal name."


User (Answer):  yash yash


Assistant (Question): "What is your primary phone number?"


User (Answer):  8899


Assistant (Question): What is your email address?


User (Answer):  hahah


Assistant (Question): Do you anticipate frequent international transactions?


User (Answer):  yes



Assistant: Thank you! We have all the information we need for onboarding.
Your collected answers: {'Please provide your full legal name.': 'yash yash', 'What is your primary phone number?': '8899', 'What is your email address?': 'hahah', 'Do you anticipate frequent international transactions?': 'yes'}


In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

# Define the ConversationalLLM class for Flan-T5
class ConversationalLLM:
    """
    Wraps an instruction-tuned encoder-decoder model (Flan-T5) for conversational use.
    """
    def __init__(self, model_name: str = "google/flan-t5-base", max_length: int = 128, temperature: float = 0.3):
        self.model_name = model_name
        self.max_length = max_length
        self.temperature = temperature
        set_seed(42)
        print(f"Loading tokenizer for {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        print(f"Loading model {model_name}...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.model.to(self.device)

    def generate_response(self, instruction: str, conversation_history: str = "") -> str:
        """
        Generate a response based on the conversation context and an instruction.
        """
        prompt = f"CONTEXT:\n{conversation_history}\n\nUSER'S TASK:\n{instruction}\n"
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_length=self.max_length,
                temperature=self.temperature,
                top_p=0.9,
                do_sample=True,
                num_return_sequences=1
            )
        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        return generated_text.strip()

# Define the ConversationManager class
class ConversationManager:
    """
    Manages the conversation flow and stores user onboarding data.
    """
    def __init__(self):
        # Define some base questions that help gather essential info.
        self.base_questions = [
            "Please provide your full legal name.",
            "What is your primary phone number?",
            "What is your email address?",
            "Do you anticipate frequent international transactions?",
        ]
        self.current_question_index = 0
        self.answers = {}
        self.conversation_history = ""

    def get_next_base_question(self):
        if self.current_question_index < len(self.base_questions):
            return self.base_questions[self.current_question_index]
        return None

    def record_answer(self, question: str, answer: str):
        self.answers[question] = answer
        self.conversation_history += f"\nUser: {answer}"

    def advance_flow(self):
        self.current_question_index += 1

    def is_onboarding_complete(self) -> bool:
        return self.current_question_index >= len(self.base_questions)

# --- Extended Conversation Loop for Fraud Detection ---
# Instantiate the LLM (Flan-T5) and the ConversationManager
llm = ConversationalLLM(model_name="google/flan-t5-base", max_length=128, temperature=0.3)
conv_manager = ConversationManager()

# Define system instructions with extra context on fraud detection.
system_instructions = (
    "You are a smart AI assistant for bank onboarding. In addition to gathering user details, "
    "your goal is to ask follow-up questions that might reveal inconsistencies or suspicious behavior. "
    "Ask questions in a friendly but probing manner, similar to an experienced investigator."
)

print("System: Onboarding conversation started.")
print("---------------------------------------")

# Loop through the base questions.
while not conv_manager.is_onboarding_complete():
    # Get the next base question.
    question = conv_manager.get_next_base_question()
    if question is None:
        break

    # Rephrase the base question in a friendly tone.
    conversation_so_far = f"System: {system_instructions}\n{conv_manager.conversation_history}"
    rephrase_instruction = f"Rephrase exactly this question in a friendly manner: \"{question}\""
    rephrased_question = llm.generate_response(
        instruction=rephrase_instruction,
        conversation_history=conversation_so_far
    )
    if len(rephrased_question.strip()) < 3 or rephrased_question.strip().isdigit():
        rephrased_question = question

    print("Assistant (Base Question):", rephrased_question)
    user_input = input("User (Answer): ")
    conv_manager.record_answer(question, user_input)
    
    # OPTIONAL: Ask a follow-up question if the answer seems vague or if further details are needed.
    # Here we prompt the model to generate a follow-up that probes for more information.
    followup_instruction = (
        f"Based on the user's answer \"{user_input}\", generate one probing follow-up question "
        "that might help detect any inconsistencies or potential fraud. "
        "The question should be friendly, natural, and aim to gather additional details."
    )
    followup_question = llm.generate_response(
        instruction=followup_instruction,
        conversation_history=conv_manager.conversation_history
    )
    # Only ask the follow-up if it appears meaningful (e.g., longer than 5 words).
    if len(followup_question.split()) > 5 and not followup_question.strip().isdigit():
        print("Assistant (Follow-up):", followup_question)
        followup_answer = input("User (Follow-up Answer): ")
        # You can choose to store follow-up answers under a special key or append to history.
        conv_manager.conversation_history += f"\nUser (Follow-up): {followup_answer}"
    
    conv_manager.advance_flow()

print("\nAssistant: Thank you! We have all the information we need for onboarding.")
print("Your collected answers:", conv_manager.answers)


Loading tokenizer for google/flan-t5-base...
Loading model google/flan-t5-base...
Using device: cpu
System: Onboarding conversation started.
---------------------------------------
Assistant (Base Question): What is the name of the person who will be assisting you?


KeyboardInterrupt: Interrupted by user

In [12]:
import openai

# Set your OpenAI API key (replace with your actual key)
openai.api_key = "YOUR_API_KEY"

# Define a ConversationalLLM class that uses OpenAI's ChatCompletion API
class ConversationalLLM:
    def __init__(self, model: str = "gpt-3.5-turbo", temperature: float = 0.3, max_tokens: int = 150):
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate_response(self, instruction: str, conversation_history: str = "") -> str:
        """
        Generate a response based on the instruction and conversation history.
        """
        messages = [
            {"role": "system", "content": conversation_history},
            {"role": "user", "content": instruction}
        ]
        response = openai.ChatCompletion.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            top_p=0.9,
            n=1  # number of responses
        )
        return response.choices[0].message['content'].strip()

# Define a ConversationManager class to manage onboarding questions and answers.
class ConversationManager:
    def __init__(self):
        self.required_questions = [
            "Please provide your full legal name.",
            "What is your primary phone number?",
            "What is your email address?",
            "Do you anticipate frequent international transactions?",
        ]
        self.current_question_index = 0
        self.answers = {}
        self.conversation_history = ""

    def get_next_question(self):
        if self.current_question_index < len(self.required_questions):
            return self.required_questions[self.current_question_index]
        return None

    def record_answer(self, question: str, answer: str):
        self.answers[question] = answer
        self.conversation_history += f"\nUser: {answer}"

    def advance_flow(self):
        self.current_question_index += 1

    def is_onboarding_complete(self) -> bool:
        return self.current_question_index >= len(self.required_questions)

# --- Conversation Loop using OpenAI's GPT-3.5 ---
llm = ConversationalLLM(model="gpt-3.5-turbo", temperature=0.3, max_tokens=150)
conv_manager = ConversationManager()

# Define system instructions to provide context to the model.
system_instructions = (
    "You are an AI assistant helping a user with bank onboarding. "
    "Gather their personal and account usage details. Be concise and user-friendly."
)

print("System: Onboarding conversation started.")
print("---------------------------------------")

while not conv_manager.is_onboarding_complete():
    # Retrieve the next base question.
    question = conv_manager.get_next_question()
    if question is None:
        break

    # Build conversation context.
    conversation_so_far = f"System: {system_instructions}\n{conv_manager.conversation_history}"
    
    # Rephrase the question using GPT-3.5.
    rephrase_instruction = f"Rephrase exactly this question in a friendly manner: \"{question}\""
    rephrased_question = llm.generate_response(instruction=rephrase_instruction, conversation_history=conversation_so_far)
    
    # Fallback to the original question if the response seems off.
    if len(rephrased_question.strip()) < 3 or rephrased_question.strip().isdigit():
        rephrased_question = question

    print("Assistant (Question):", rephrased_question)
    
    # Capture user input in the notebook.
    user_input = input("User (Answer): ")
    
    # Record the answer and advance the conversation flow.
    conv_manager.record_answer(question, user_input)
    conv_manager.advance_flow()

print("\nAssistant: Thank you! We have all the information we need for onboarding.")
print("Your collected answers:", conv_manager.answers)


System: Onboarding conversation started.
---------------------------------------


APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742
